- Delete Duplicate
- Column Schema Evolution
- DataTypes Handleing / Type casting
- Null Records Handling
- Triming
- Case Conversion
- Maintain Order of Data

In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)


In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"
# print(silverTablName)
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawsales"
# print(bronzeTablName)
srcFileLoc=f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/daily_sales/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTablName} 
SELECT DISTINCT
    CAST(TRIM(sale_id) AS INTEGER) as sale_id ,
    UPPER(TRIM(product)) as product,
    UPPER(TRIM(category)) as category,
    CAST(TRIM(quantity) AS INTEGER) as quantity,
    CAST(TRIM(price) AS DOUBLE) as price,
    TO_DATE(TRIM(sale_date),'yyyy-MM-dd') as sale_date,-- 2024-04-08
    UPPER(TRIM(region)) as region,
    CURRENT_TIMESTAMP() as ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                    FROM {silverTablName}
                    )
ORDER BY CAST(TRIM(sale_id) AS INTEGER)
""")

In [0]:
# %sql
# SELECT * FROM intellibi_catlog.intellibi_bronzE.fMonthlySales;
# %sql
# SELECT * FROM intellibi_catlog.intellibi_silver.fMonthlySales;
# %sql
# SELECT * FROM intellibi_catlog.intellibi_SILVER.fMonthlySales;
